# 2D Darcy Flow with FNO2D

Learn the map from a permeability field $a(x)$ to the pressure field $u(x)$ of steady flow through
porous media with a 2D Fourier Neural Operator ([Li et al., 2021](https://arxiv.org/abs/2010.08895)).
The pressure solves

$$-\nabla \cdot (a(x) \nabla u(x)) = f(x), \quad x \in [0, 1]^2, \quad u|_{\partial \Omega} = 0.$$

Once trained, the FNO predicts the pressure for a new permeability field in one forward pass instead
of running a PDE solver.

In [ ]:
# Install keras-sciml, then choose a backend: "tensorflow", "torch" or "jax"
!pip install keras-sciml
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Hyperparameters

`MODES` is the number of Fourier modes kept in each spatial dimension.

In [ ]:
N_SAMPLES = 64
RESOLUTION = 16
MODES = 8
WIDTH = 32
DEPTH = 3
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.01

## Load the data

`DarcyFlowDataset` draws piecewise-constant permeability fields and solves for the pressure on a
`RESOLUTION x RESOLUTION` grid. 80% of the samples are used for training and the rest for testing.

In [ ]:
import keras

from keras_sciml.datasets import DarcyFlowDataset
from keras_sciml.losses import RelativeL2Loss
from keras_sciml.models import FNO2D

keras.utils.set_random_seed(42)

ds = DarcyFlowDataset(n_samples=N_SAMPLES, resolution=RESOLUTION, field_type="piecewise", seed=42)
a_data, u_data = ds.as_tuple()

n_train = int(0.8 * N_SAMPLES)
a_train, u_train = a_data[:n_train], u_data[:n_train]
a_test, u_test = a_data[n_train:], u_data[n_train:]
print(f"Train: {a_train.shape}, Test: {a_test.shape}")

## Define the model

`FNO2D` lifts the permeability to `WIDTH` channels, applies `DEPTH` spectral convolution blocks, and
projects back to one pressure channel. The relative $L^2$ loss
$\|u_{\text{pred}} - u_{\text{true}}\|_2 / \|u_{\text{true}}\|_2$ weights every sample equally,
whatever its magnitude.

In [ ]:
model = FNO2D(
    modes=(MODES, MODES),
    width=WIDTH,
    depth=DEPTH,
    out_channels=1,
    activation="gelu",
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=RelativeL2Loss(),
    metrics=[RelativeL2Loss(name="rel_l2")],
)

## Train

In [ ]:
history = model.fit(
    a_train,
    u_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(a_test, u_test),
    verbose=2,
)

## Evaluate

In [ ]:
test_loss = float(model.evaluate(a_test, u_test, verbose=0)[0])
print(f"Test relative L2 error: {test_loss:.4f}")